<img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png"  width='15%'>  


<h1> Demo: PEFT Instruction Tuning</h1>
Developed by WeCloudData
<br></br>




Large Language Models (LLMs) are powerful, but they are trained to be general-purpose. In real applications—such as chatbots, customer support, or internal knowledge systems , we need models that understand our domain, tone, and data.

Fully fine-tuning an LLM is expensive and slow because it requires updating billions of parameters.
PEFT (Parameter-Efficient Fine-Tuning) solves this by freezing the base model and training only a small set of lightweight adapter weights. This makes customization fast, affordable, and practical.

In this notebook, we will use a PEFT method (such as LoRA) to adapt a pretrained model and observe how it becomes better aligned with our target task—without retraining the entire model.

In [1]:
# Simple PEFT Instruction Tuning Demo for Google Colab
# Uses a small model (GPT-2) with focused training data

# Install dependencies
!pip install -q transformers datasets peft accelerate bitsandbytes
import sys
!{sys.executable} -m pip install -q "torchao>=0.16.0"

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType

# ============================================================================
# 1. CREATE A BETTER INSTRUCTION DATASET
# ============================================================================

# Focused on Q&A about general knowledge - easier for model to learn
# More examples, consistent format, better quality
data = {
    "instruction": [
        # Geography questions
        "What is the capital of France?",
        "What is the capital of Japan?",
        "What is the capital of Italy?",
        "What is the capital of Germany?",
        "What is the capital of Spain?",
        "What is the capital of Canada?",
        "What is the capital of Australia?",
        "What is the capital of Brazil?",

        # Simple math
        "What is 10 + 15?",
        "What is 25 + 30?",
        "What is 50 + 25?",
        "What is 100 - 40?",
        "What is 200 - 75?",
        "What is 12 * 5?",
        "What is 8 * 7?",
        "What is 100 / 4?",

        # Basic facts
        "How many days are in a week?",
        "How many months are in a year?",
        "How many hours are in a day?",
        "How many minutes are in an hour?",
        "How many continents are there?",
        "How many colors are in a rainbow?",
        "How many sides does a triangle have?",
        "How many legs does a spider have?",

        # Simple definitions
        "What is the sun?",
        "What is water made of?",
        "What is oxygen?",
        "What is gravity?",
        "What is the internet?",
        "What is a computer?",
        "What is electricity?",
        "What is photosynthesis?",

        # Yes/No questions
        "Is the Earth round?",
        "Is the sun a star?",
        "Is water wet?",
        "Is ice cold?",
        "Can birds fly?",
        "Can fish breathe underwater?",
        "Do plants need sunlight?",
        "Is Python a programming language?",
    ],
    "output": [
        # Geography answers
        "The capital of France is Paris.",
        "The capital of Japan is Tokyo.",
        "The capital of Italy is Rome.",
        "The capital of Germany is Berlin.",
        "The capital of Spain is Madrid.",
        "The capital of Canada is Ottawa.",
        "The capital of Australia is Canberra.",
        "The capital of Brazil is Brasília.",

        # Math answers
        "10 + 15 = 25",
        "25 + 30 = 55",
        "50 + 25 = 75",
        "100 - 40 = 60",
        "200 - 75 = 125",
        "12 * 5 = 60",
        "8 * 7 = 56",
        "100 / 4 = 25",

        # Basic facts answers
        "There are 7 days in a week.",
        "There are 12 months in a year.",
        "There are 24 hours in a day.",
        "There are 60 minutes in an hour.",
        "There are 7 continents.",
        "There are 7 colors in a rainbow.",
        "A triangle has 3 sides.",
        "A spider has 8 legs.",

        # Definition answers
        "The sun is a star at the center of our solar system that provides light and heat to Earth.",
        "Water is made of two hydrogen atoms and one oxygen atom (H2O).",
        "Oxygen is a gas that humans and animals need to breathe to survive.",
        "Gravity is a force that pulls objects toward each other, keeping us on the ground.",
        "The internet is a global network of computers that allows people to share information.",
        "A computer is an electronic device that processes data and performs tasks.",
        "Electricity is a form of energy that powers devices and provides light and heat.",
        "Photosynthesis is the process plants use to convert sunlight into energy.",

        # Yes/No answers
        "Yes, the Earth is round (spherical).",
        "Yes, the sun is a star.",
        "Yes, water is wet.",
        "Yes, ice is cold.",
        "Yes, most birds can fly.",
        "Yes, fish can breathe underwater using gills.",
        "Yes, plants need sunlight to grow.",
        "Yes, Python is a programming language.",
    ]
}

dataset = Dataset.from_dict(data)

# Split into train/test (80/20 split)
dataset = dataset.train_test_split(test_size=0.2, seed=42)

print(f"✅ Training samples: {len(dataset['train'])}")
print(f"✅ Test samples: {len(dataset['test'])}")
print(f"\nExample:\nQ: {dataset['train'][0]['instruction']}\nA: {dataset['train'][0]['output']}\n")

# ============================================================================
# 2. LOAD MODEL AND TOKENIZER (GPT-2 Small - only 124M parameters!)
# ============================================================================

model_name = "gpt2"  # Tiny model, perfect for demos
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Set pad token
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

# ============================================================================
# 3. FORMAT DATA AS INSTRUCTION-RESPONSE PAIRS
# ============================================================================

def format_instruction(sample):
    """Format data in instruction-following format"""
    instruction = sample["instruction"]
    output = sample["output"]

    # Simple, clean format
    prompt = f"Question: {instruction}\nAnswer: {output}"

    return {"text": prompt + tokenizer.eos_token}

# Apply formatting
dataset = dataset.map(format_instruction, remove_columns=dataset["train"].column_names)

# Tokenize
def tokenize(sample):
    result = tokenizer(
        sample["text"],
        truncation=True,
        max_length=128,  # Shorter since our examples are concise
        padding=False
    )
    result["labels"] = result["input_ids"].copy()
    return result

tokenized_dataset = dataset.map(tokenize, remove_columns=["text"], batched=False)

# ============================================================================
# 4. CONFIGURE LoRA PEFT
# ============================================================================

lora_config = LoraConfig(
    r=8,                    # Rank of the low-rank matrices
    lora_alpha=16,          # Scaling factor
    target_modules=["c_attn"],  # GPT-2 uses c_attn for Q,K,V projections
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # Shows only ~0.3% parameters are trainable!

# ============================================================================
# 5. TRAINING CONFIGURATION
# ============================================================================

training_args = TrainingArguments(
    output_dir="./instruction_tuned_gpt2",
    num_train_epochs=10,  # More epochs since we have more data
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=3e-4,
    fp16=True,
    logging_steps=10,
    eval_strategy="epoch",  # Changed from evaluation_strategy for newer transformers
    save_strategy="epoch",
    load_best_model_at_end=True,
    warmup_steps=20,
    report_to="none",
    weight_decay=0.01,
)

# Custom data collator to ensure proper padding
from dataclasses import dataclass
from typing import Any, Dict, List

@dataclass
class DataCollatorForCompletionOnlyLM:
    tokenizer: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        # Extract input_ids and labels
        input_ids = [f["input_ids"] for f in features]
        labels = [f["labels"] for f in features]

        # Find max length in batch
        max_length = max(len(ids) for ids in input_ids)

        # Pad sequences
        padded_input_ids = []
        padded_labels = []
        attention_mask = []

        for ids, lbls in zip(input_ids, labels):
            padding_length = max_length - len(ids)

            # Pad input_ids and attention_mask
            padded_input_ids.append(ids + [self.tokenizer.pad_token_id] * padding_length)
            attention_mask.append([1] * len(ids) + [0] * padding_length)

            # Pad labels with -100 (ignored in loss calculation)
            padded_labels.append(lbls + [-100] * padding_length)

        return {
            "input_ids": torch.tensor(padded_input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(padded_labels, dtype=torch.long)
        }

data_collator = DataCollatorForCompletionOnlyLM(tokenizer=tokenizer)

# ============================================================================
# 6. TRAIN THE MODEL
# ============================================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
)

print("\n🚀 Starting training...\n")
trainer.train()

# ============================================================================
# 7. TEST THE FINE-TUNED MODEL
# ============================================================================

def generate_response(question):
    """Generate a response for a given question"""
    prompt = f"Question: {question}\nAnswer:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.2
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract just the answer part
    if "Answer:" in response:
        response = response.split("Answer:")[-1].strip()
    return response

print("\n" + "="*60)
print("🎯 TESTING THE FINE-TUNED MODEL")
print("="*60 + "\n")

# Test with examples from training distribution
test_cases = [
    "What is the capital of France?",  # From training
    "What is the capital of Mexico?",  # Similar to training
    "What is 15 + 20?",  # From training pattern
    "How many days are in a week?",  # From training
    "Is the moon a planet?",  # Similar to training pattern
    "What is the capital of England?",  # New but similar
]

for question in test_cases:
    print(f"❓ Question: {question}")
    print(f"💡 Answer: {generate_response(question)}")
    print("-" * 60)

# ============================================================================
# 8. SAVE THE MODEL
# ============================================================================

model.save_pretrained("./instruction_tuned_gpt2_final")
tokenizer.save_pretrained("./instruction_tuned_gpt2_final")

print("\n✅ Model saved! You can load it later with:")
print("   from peft import PeftModel, AutoPeftModelForCausalLM")
print("   model = AutoPeftModelForCausalLM.from_pretrained('./instruction_tuned_gpt2_final')")

# ============================================================================
# 9. (OPTIONAL) COMPARE WITH BASE MODEL
# ============================================================================

print("\n" + "="*60)
print("🔍 COMPARISON: Base Model vs Fine-tuned")
print("="*60 + "\n")

# Load base model for comparison
base_model = AutoModelForCausalLM.from_pretrained("gpt2")
base_model.eval()

test_question = "What is the capital of France?"
prompt = f"Question: {test_question}\nAnswer:"

print(f"❓ Question: {test_question}\n")

# Base model response
inputs = tokenizer(prompt, return_tensors="pt")
with torch.no_grad():
    base_output = base_model.generate(
        **inputs,
        max_new_tokens=50,
        pad_token_id=tokenizer.eos_token_id,
        temperature=0.7,
        do_sample=True
    )
base_response = tokenizer.decode(base_output[0], skip_special_tokens=True)

print(f"❌ Base Model Response:\n{base_response}\n")
print(f"✅ Fine-tuned Model Response:\n{generate_response(test_question)}\n")

print("="*60)
print("🎉 Demo complete!")
print("   The fine-tuned model should give focused, direct answers")
print("   while the base model may ramble or go off-topic.")
print("="*60)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 27.7 MB/s eta 0:00:00


✅ Training samples: 32
✅ Test samples: 8

Example:
Q: What is water made of?
A: Water is made of two hydrogen atoms and one oxygen atom (H2O).



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

...

Epoch,Training Loss,Validation Loss
1,No log,2.696406
2,No log,2.673040
3,3.138110,2.615147
4,3.138110,2.517680
5,3.001608,2.306550
6,3.001608,2.046918
7,3.001608,1.858912
8,2.570899,1.741079
9,2.570899,1.673151
10,2.168482,1.648247



🎯 TESTING THE FINE-TUNED MODEL

❓ Question: What is the capital of France?
💡 Answer: The Capital (French) is French for "Capital".
------------------------------------------------------------
❓ Question: What is the capital of Mexico?
💡 Answer: The Capital Is, in Spanish.
------------------------------------------------------------
❓ Question: What is 15 + 20?
💡 Answer: The answer to this question depends on the value of 16. A given number can be written as 14 plus 12 or 17 minus 6 (or 9) in English and French, respectively; it's possible to write a sentence that represents one point with an
------------------------------------------------------------
❓ Question: How many days are in a week?
💡 Answer: Two. In our most recent survey of American adults, we asked about the length and duration each month for two consecutive months (September-October). Our results show that average weekly lengths were 5 weeks while shortest is 1 day . This means there was
----------------------------------

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


❓ Question: What is the capital of France?

❌ Base Model Response:
Question: What is the capital of France?
Answer: The capital of France is 100 million francs.
French politicians think that the French capital is the capital of the French Republic.
You can be assured that this is a contradiction. France was founded in 1858.
In 1858, the

✅ Fine-tuned Model Response:
The French are known as "Frenchmen". They were born in 1711 and became citizens on April 4, 1601. In 1802 they came to England after a long period of exile from Paris where he was raised by his mother's family until

🎉 Demo complete!
   The fine-tuned model should give focused, direct answers
   while the base model may ramble or go off-topic.


In [2]:
# Simple PEFT Instruction Tuning Demo for Google Colab
# Uses a small model (GPT-2) with focused training data

# Install dependencies
!pip install -q transformers datasets peft accelerate bitsandbytes

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType

# ============================================================================
# 1. CREATE A BETTER INSTRUCTION DATASET
# ============================================================================

# Focused on Q&A about general knowledge - easier for model to learn
# More examples, consistent format, better quality
data = {
    "instruction": [
        # Basic bootcamp information
        "What is the bootcamp about?",
        "Where is the bootcamp located?",
        "When did Week 0 sessions take place?",
        "What time do the daily sessions start?",
        "What time do the daily sessions end?",
        "What is the timezone for the bootcamp?",
        "Is attendance mandatory?",
        "What is the Zoom link for the bootcamp?",

        # Week 0 activities
        "What surveys were students required to complete in Week 0?",
        "What is the purpose of the student background survey?",
        "What does the AI skills and goals survey assess?",
        "Where can students submit questions or issues?",
        "What was the class schedule for Week 0?",

        # Project-related questions
        "What is the capstone project about?",
        "When did students start working on the capstone project?",
        "What are some project ideas mentioned?",
        "How many groups were formed for projects?",
        "When is the project proposal due?",
        "What should be included in the project proposal?",
        "How are project proposals submitted?",

        # Project ideas
        "What are the AI for Education project focuses?",
        "What is the AI for RFP project about?",
        "What is the AI for HR project about?",
        "What is the Enterprise AI Hub project?",
        "What is the Apache Doris/VeloDB project?",

        # Technical topics covered
        "What Python topics were covered in Week 1?",
        "What is covered in the Pandas module?",
        "What machine learning topics were taught?",
        "What NLP topics were introduced?",
        "What library is used for sentence embeddings?",

        # Instructors and support
        "Who are the instructors mentioned?",
        "How can students get help during the bootcamp?",
        "Where can students find office hours?",
        "How do students provide instructor feedback?",

        # Schedule and holidays
        "What holiday occurred during the bootcamp?",
        "When was Saudi National Day?",
        "What week did the bootcamp officially start the capstone?",

        # Technical tools and platforms
        "What platform is used for learning materials?",
        "What tools are used for collaboration?",
        "What is the Miro board used for?",
        "Where can students find open data for Saudi Arabia?",

        # Code examples
        "How do you flatten a nested list in Python?",
        "How do you use OpenAI's API in Python?",
        "What is the format for chat history in OpenAI?",
        "How do you create embeddings with sentence transformers?",

        # Project validation
        "How should teams conduct user validation?",
        "When should user validation surveys be created?",
        "Where do teams submit user validation links?",

        # Presentation requirements
        "When is the project proposal presentation?",
        "Is the first presentation graded?",
        "How many presentations are expected during the program?",

        # Model information
        "What is Claude Sonnet 4.5?",
        "What Claude models are available?",
        "How can you access Claude?",

        # Additional resources
        "Where can students find the LLM from scratch book?",
        "What is Modal and what is it used for?",
        "Where can students find the Google ROI of AI report?",

        # Client projects
        "What clients are available for the capstone project?",
        "When do client projects officially start?",
        "Can students change their project team?",
    ],
    "output": [
        # Basic bootcamp information
        "This is a GenAI MLOps/LLMOps Bootcamp organized by WeCloudData and SDA (Saudi Digital Academy) in Riyadh.",
        "The bootcamp is located in Riyadh, Saudi Arabia.",
        "Week 0 sessions took place from Monday, August 25 to Thursday, August 28, 2025.",
        "Daily sessions start at 9:00 AM Riyadh time.",
        "Daily sessions end at 12:00 PM Riyadh time (with some afternoon sessions extending to 5:00 PM).",
        "The bootcamp operates in Riyadh Time timezone.",
        "Yes, attendance is mandatory and is taken daily.",
        "The main Zoom link is https://us02web.zoom.us/j/83424596656 with passcode 726039.",

        # Week 0 activities
        "Students were required to complete two surveys: a student background survey and an AI skills and goals survey.",
        "The student background survey helps understand students' background, motivations, and technical familiarity to tailor the bootcamp experience.",
        "The AI skills and goals survey assesses students' current AI skills and learning objectives.",
        "Students can submit questions or issues in a shared Google spreadsheet for office hour tickets.",
        "Week 0 sessions ran Monday through Thursday (Aug 25-28, 2025) from 9:00 AM to 12:00 PM.",

        # Project-related questions
        "The capstone project involves developing AI solutions in areas like Education, RFP, HR, or Enterprise AI, working in teams to solve real-world problems.",
        "Students started researching capstone projects on September 1, 2025 (Week 1).",
        "Project ideas include: AI for Education (instructors, students, schools), AI for Sales/RFP, AI for HR (workforce planning, hiring, job applications), Enterprise AI Hub, Apache Doris/VeloDB QA chatbot, and Text to SQL for real-time data warehouse.",
        "There were 10 project groups formed (Group 1 through Group 10).",
        "The draft project proposal and presentation were due at the end of Week 2, with presentations on Thursday of Week 2.",
        "Proposals should include: pain points, market research, user research via class survey, proposed solution (AI product description and implementation plan), and can be in Word doc or presentation format.",
        "Project proposals are submitted through a Google Form, with teams also maintaining collaborative Google Docs.",

        # Project ideas
        "AI for Education focuses on creating solutions for instructors, students, and schools to improve learning experiences.",
        "AI for RFP (Request for Proposal) aims to automate and streamline the RFP response process using AI.",
        "AI for HR includes workforce planning, hiring/recruiting, and job application assistance.",
        "Enterprise AI Hub focuses on secure deployment of LLM, Chat, Knowledge Search, and Workflows for enterprises.",
        "Apache Doris/VeloDB project involves creating a QA chatbot and Text to SQL functionality for real-time data warehouses.",

        # Technical topics covered
        "Week 1 covered Python fundamentals including functions, nested lists, list comprehension, and recursion.",
        "The Pandas module covers data manipulation, DataFrames, SQL-style joins, and data analysis techniques.",
        "Machine learning topics include regression, ensemble trees (bagging), hyperparameter tuning, and credit risk modeling.",
        "NLP topics include text preprocessing, tokenization, stemming, TF-IDF vectorization, sentiment analysis, and sentence embeddings.",
        "SentenceTransformer library (e.g., 'all-MiniLM-L6-v2' model) is used for creating sentence embeddings.",

        # Instructors and support
        "Instructors mentioned include WCD-Idris, shaohua-weclouddata, WCD-Carlo, Esteban (Gen AI - WCD), and Prabesh Lamichhane.",
        "Students can get help by using the office hour tickets spreadsheet, asking in class, or through breakout rooms during project sessions.",
        "Office hours can be accessed through the Week 0 Office Hour Tickets Google spreadsheet.",
        "Students provide instructor feedback through anonymous Google Forms surveys, conducted multiple times during the bootcamp.",

        # Schedule and holidays
        "Tuesday, September 23rd, 2025 was a holiday for Saudi National Day.",
        "Saudi National Day was celebrated on September 23rd, 2025.",
        "The capstone project officially starts in Week 9 or Week 10, though preparation began in Week 1.",

        # Technical tools and platforms
        "The learning portal at sda.weclouddata.com is used for accessing course materials, videos, and assignments.",
        "Tools used include Google Docs for collaboration, Miro boards for group ideation, Google Forms for surveys, and Zoom for virtual sessions.",
        "The Miro board is used for group assignment, project ideation, and organizing team information.",
        "Open data for Saudi Arabia can be found at https://data.gov.sa/ar, the National Data Bank portal.",

        # Code examples
        "You can flatten a nested list using recursion: check if each element is a list using isinstance(), recursively call the function on list elements, and append non-list elements to the result.",
        "To use OpenAI's API: import openai, create a client with openai.OpenAI(api_key=api_key), then call client.chat.completions.create() with model and messages parameters.",
        "Chat history format is a list of dictionaries with 'role' and 'content' keys: [{'role': 'user', 'content': 'question'}, {'role': 'assistant', 'content': 'answer'}].",
        "To create embeddings: from sentence_transformers import SentenceTransformer, load model with SentenceTransformer('all-MiniLM-L6-v2'), then call model.encode(sentences).",

        # Project validation
        "Teams should create user validation surveys, have classmates complete them, and track responses to validate their project ideas.",
        "User validation surveys should be created on Sunday (Week 2, Day 7) and distributed on Monday for completion.",
        "Teams submit and track user validation links in a shared Google spreadsheet.",

        # Presentation requirements
        "The project proposal presentation was held on Thursday of Week 2 (September 11, 2025).",
        "No, the first presentation in Week 2 was a practice presentation and was not graded.",
        "Students are expected to do 2-3 more presentations throughout the program, which will be graded.",

        # Model information
        "Claude Sonnet 4.5 is the smartest model in the Claude 4 family, efficient for everyday use, created by Anthropic.",
        "The Claude 4 family consists of Claude Opus 4.1, Claude Opus 4, Claude Sonnet 4.5, and Claude Sonnet 4.",
        "Claude can be accessed via web-based chat interface, mobile and desktop apps, API and developer platform (model string 'claude-sonnet-4-5-20250929'), and Claude Code command line tool.",

        # Additional resources
        "The book 'Build a Large Language Model (From Scratch)' by Sebastian Raschka is available as a shared PDF resource.",
        "Modal (modal.com) is a platform that provides free GPU access (smaller GPUs) for high-performance AI infrastructure and serverless computing.",
        "The Google Cloud ROI of AI 2025 report is available as a shared resource, emphasizing thinking about problems rather than just ideas.",

        # Client projects
        "Available clients include: WeCloudData (for Edtech AI projects), Beamdata (for MLOps/LLMOps projects), and VeloDB (starting Week 9).",
        "Client projects officially start in Week 9 or Week 10, though preparation and research begin earlier.",
        "Yes, students don't have to stay in the same project team for their final capstone project. Re-grouping happens when the real client project starts after Week 7.",
    ]
}

dataset = Dataset.from_dict(data)

# Split into train/test (80/20 split)
dataset = dataset.train_test_split(test_size=0.2, seed=42)

print(f"✅ Training samples: {len(dataset['train'])}")
print(f"✅ Test samples: {len(dataset['test'])}")
print(f"\nExample:\nQ: {dataset['train'][0]['instruction']}\nA: {dataset['train'][0]['output']}\n")

# ============================================================================
# 2. LOAD MODEL AND TOKENIZER (Qwen2.5 - Modern SLM!)
# ============================================================================

model_name = "Qwen/Qwen2.5-0.5B-Instruct"  # You can also try "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Set pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

# ============================================================================
# 3. FORMAT DATA AS INSTRUCTION-RESPONSE PAIRS
# ============================================================================

def format_instruction(sample):
    """Format data in instruction-following format"""
    instruction = sample["instruction"]
    output = sample["output"]

    # Simple, clean format
    prompt = f"Question: {instruction}\nAnswer: {output}"

    return {"text": prompt + tokenizer.eos_token}

# Apply formatting
dataset = dataset.map(format_instruction, remove_columns=dataset["train"].column_names)

# Tokenize
def tokenize(sample):
    result = tokenizer(
        sample["text"],
        truncation=True,
        max_length=128,  # Shorter since our examples are concise
        padding=False
    )
    result["labels"] = result["input_ids"].copy()
    return result

tokenized_dataset = dataset.map(tokenize, remove_columns=["text"], batched=False)

# ============================================================================
# 4. CONFIGURE LoRA PEFT
# ============================================================================

lora_config = LoraConfig(
    r=8,                    # Rank of the low-rank matrices
    lora_alpha=16,          # Scaling factor
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # Qwen2.5 attention modules
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # Shows only ~0.3% parameters are trainable!

# ============================================================================
# 5. TRAINING CONFIGURATION
# ============================================================================

training_args = TrainingArguments(
    output_dir="./instruction_tuned_qwen0.5",
    num_train_epochs=10,  # More epochs since we have more data
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=3e-4,
    fp16=True,
    logging_steps=10,
    eval_strategy="epoch",  # Changed from evaluation_strategy for newer transformers
    save_strategy="epoch",
    load_best_model_at_end=True,
    warmup_steps=20,
    report_to="none",
    weight_decay=0.01,
)

# Custom data collator to ensure proper padding
from dataclasses import dataclass
from typing import Any, Dict, List

@dataclass
class DataCollatorForCompletionOnlyLM:
    tokenizer: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        # Extract input_ids and labels
        input_ids = [f["input_ids"] for f in features]
        labels = [f["labels"] for f in features]

        # Find max length in batch
        max_length = max(len(ids) for ids in input_ids)

        # Pad sequences
        padded_input_ids = []
        padded_labels = []
        attention_mask = []

        for ids, lbls in zip(input_ids, labels):
            padding_length = max_length - len(ids)

            # Pad input_ids and attention_mask
            padded_input_ids.append(ids + [self.tokenizer.pad_token_id] * padding_length)
            attention_mask.append([1] * len(ids) + [0] * padding_length)

            # Pad labels with -100 (ignored in loss calculation)
            padded_labels.append(lbls + [-100] * padding_length)

        return {
            "input_ids": torch.tensor(padded_input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(padded_labels, dtype=torch.long)
        }

data_collator = DataCollatorForCompletionOnlyLM(tokenizer=tokenizer)

# ============================================================================
# 6. TRAIN THE MODEL
# ============================================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
)

print("\n🚀 Starting training...\n")
trainer.train()

# ============================================================================
# 7. TEST THE FINE-TUNED MODEL
# ============================================================================

def generate_response(question):
    """Generate a response for a given question"""
    prompt = f"Question: {question}\nAnswer:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.2
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract just the answer part
    if "Answer:" in response:
        response = response.split("Answer:")[-1].strip()
    return response

print("\n" + "="*60)
print("🎯 TESTING THE FINE-TUNED MODEL")
print("="*60 + "\n")

# Test with examples from training distribution

test_cases = [
    # Direct matches from training (5 cases)
    "What is the bootcamp about?",  # From training
    "Where is the bootcamp located?",  # From training
    "Is attendance mandatory?",  # From training
    "What are some project ideas mentioned?",  # From training
    "What Python topics were covered in Week 1?",  # From training

    # Similar patterns to training (10 cases)
    "What time does the bootcamp start each day?",  # Similar to "What time do sessions start?"
    "How many surveys did students need to complete?",  # Similar to survey questions
    "Who can students contact for help?",  # Similar to support questions
    "When do students present their proposals?",  # Similar to presentation timing
    "What is the purpose of the capstone project?",  # Similar to project questions
    "Which tools are used for team collaboration?",  # Similar to tools questions
    "What happened on September 23rd, 2025?",  # Similar to holiday questions
    "How do you create a nested list in Python?",  # Similar to code questions
    "What machine learning algorithms were taught?",  # Similar to technical topics
    "Can students switch project teams later?",  # Similar to team questions

    # New but related to bootcamp domain (5 cases)
    "What programming languages are taught in the bootcamp?",  # New inference
    "How long is the entire bootcamp program?",  # Requires inference from weeks mentioned
    "What should I do if I'm sick and can't attend?",  # Combines sick leave info
    "Are there any breaks during the day?",  # Prayer break mentioned
    "What happens after Week 7 of the bootcamp?",  # Requires timeline understanding
]

for question in test_cases:
    print(f"❓ Question: {question}")
    print(f"💡 Answer: {generate_response(question)}")
    print("-" * 60)

# ============================================================================
# 8. SAVE THE MODEL
# ============================================================================

model.save_pretrained("./instruction_tuned_qwen0.5")
tokenizer.save_pretrained("./instruction_tuned_qwen0.5")

print("\n✅ Model saved! You can load it later with:")
print("   from peft import PeftModel, AutoPeftModelForCausalLM")
print("   model = AutoPeftModelForCausalLM.from_pretrained('./instruction_tuned_gpt2_final')")

# ============================================================================
# 9. (OPTIONAL) COMPARE WITH BASE MODEL
# ============================================================================

print("\n" + "="*60)
print("🔍 COMPARISON: Base Model vs Fine-tuned")
print("="*60 + "\n")

# Load base model for comparison
base_model = AutoModelForCausalLM.from_pretrained(model_name)
base_model.eval()

test_question = "What is the bootcamp about?"
prompt = f"Question: {test_question}\nAnswer:"

print(f"❓ Question: {test_question}\n")

# Base model response
inputs = tokenizer(prompt, return_tensors="pt")
with torch.no_grad():
    base_output = base_model.generate(
        **inputs,
        max_new_tokens=50,
        pad_token_id=tokenizer.pad_token_id,
        temperature=0.7,
        do_sample=True
    )
base_response = tokenizer.decode(base_output[0], skip_special_tokens=True)

print(f"❌ Base Qwen2.5 Response:\n{base_response}\n")
print(f"✅ Fine-tuned Model Response:\n{generate_response(test_question)}\n")

print("="*60)
print("🎉 Demo complete!")
print("   The fine-tuned model should give focused, direct answers")
print("   while the base model may be more verbose or unfocused.")
print("="*60)

✅ Training samples: 48
✅ Test samples: 12

Example:
Q: What clients are available for the capstone project?
A: Available clients include: WeCloudData (for Edtech AI projects), Beamdata (for MLOps/LLMOps projects), and VeloDB (starting Week 9).



config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184

🚀 Starting training...



Epoch,Training Loss,Validation Loss
1,No log,3.207723
2,3.262260,2.982229
3,3.262260,2.624016
4,2.735522,2.387912
5,2.139655,2.319171
6,2.139655,2.336074
7,1.760267,2.344077
8,1.760267,2.386910
9,1.555765,2.433788
10,1.350398,2.442227



🎯 TESTING THE FINE-TUNED MODEL

❓ Question: What is the bootcamp about?
💡 Answer: The Bootcamp focuses on building a team, learning programming languages like Python and Java, practicing projects in real-time environments (Jupyter notebooks), hands-on workshops led by industry experts, weekly meetings with mentors from top tech companies, and project management skills.
------------------------------------------------------------
❓ Question: Where is the bootcamp located?
💡 Answer: The bootcamp is located in Bangalore, India.
------------------------------------------------------------
❓ Question: Is attendance mandatory?
💡 Answer: Attendance is required for all sessions, including group work and presentations.
------------------------------------------------------------
❓ Question: What are some project ideas mentioned?
💡 Answer: There were several project idea proposals submitted, ranging from a simple text summarization to complex language modeling tasks.
---------------------------

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

❓ Question: What is the bootcamp about?

❌ Base Qwen2.5 Response:
Question: What is the bootcamp about?
Answer: The Bootcamp was a 10-day course offered by Microsoft to help people learn programming languages and software development. It was created in 2008, with the aim of providing a practical, hands-on experience for students who are interested in

✅ Fine-tuned Model Response:
The bootcamp focuses on project management and software development.

🎉 Demo complete!
   The fine-tuned model should give focused, direct answers
   while the base model may be more verbose or unfocused.


In [ ]:
# Improved PEFT Fine-tuning for Knowledge Injection
# Key fixes to ensure the model LEARNS new knowledge

!pip install -q transformers datasets peft accelerate bitsandbytes

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# ============================================================================
# KEY IMPROVEMENTS FOR KNOWLEDGE LEARNING
# ============================================================================
# 1. Use 4-bit quantization to save memory and train more parameters
# 2. Increase LoRA rank (r) for more capacity
# 3. Target MORE modules (including MLP layers)
# 4. Use completion-only training (mask the instruction part)
# 5. More training epochs and better learning rate
# 6. Add data augmentation (paraphrasing)
# ============================================================================

# ============================================================================
# 1. ENHANCED DATASET WITH AUGMENTATION
# ============================================================================

# Original data
data = {
     "instruction": [
        # Basic bootcamp information
        "What is the bootcamp about?",
        "Where is the bootcamp located?",
        "When did Week 0 sessions take place?",
        "What time do the daily sessions start?",
        "What time do the daily sessions end?",
        "What is the timezone for the bootcamp?",
        "Is attendance mandatory?",
        "What is the Zoom link for the bootcamp?",

        # Week 0 activities
        "What surveys were students required to complete in Week 0?",
        "What is the purpose of the student background survey?",
        "What does the AI skills and goals survey assess?",
        "Where can students submit questions or issues?",
        "What was the class schedule for Week 0?",

        # Project-related questions
        "What is the capstone project about?",
        "When did students start working on the capstone project?",
        "What are some project ideas mentioned?",
        "How many groups were formed for projects?",
        "When is the project proposal due?",
        "What should be included in the project proposal?",
        "How are project proposals submitted?",

        # Project ideas
        "What are the AI for Education project focuses?",
        "What is the AI for RFP project about?",
        "What is the AI for HR project about?",
        "What is the Enterprise AI Hub project?",
        "What is the Apache Doris/VeloDB project?",

        # Technical topics covered
        "What Python topics were covered in Week 1?",
        "What is covered in the Pandas module?",
        "What machine learning topics were taught?",
        "What NLP topics were introduced?",
        "What library is used for sentence embeddings?",

        # Instructors and support
        "Who are the instructors mentioned?",
        "How can students get help during the bootcamp?",
        "Where can students find office hours?",
        "How do students provide instructor feedback?",

        # Schedule and holidays
        "What holiday occurred during the bootcamp?",
        "When was Saudi National Day?",
        "What week did the bootcamp officially start the capstone?",

        # Technical tools and platforms
        "What platform is used for learning materials?",
        "What tools are used for collaboration?",
        "What is the Miro board used for?",
        "Where can students find open data for Saudi Arabia?",

        # Code examples
        "How do you flatten a nested list in Python?",
        "How do you use OpenAI's API in Python?",
        "What is the format for chat history in OpenAI?",
        "How do you create embeddings with sentence transformers?",

        # Project validation
        "How should teams conduct user validation?",
        "When should user validation surveys be created?",
        "Where do teams submit user validation links?",

        # Presentation requirements
        "When is the project proposal presentation?",
        "Is the first presentation graded?",
        "How many presentations are expected during the program?",

        # Model information
        "What is Claude Sonnet 4.5?",
        "What Claude models are available?",
        "How can you access Claude?",

        # Additional resources
        "Where can students find the LLM from scratch book?",
        "What is Modal and what is it used for?",
        "Where can students find the Google ROI of AI report?",

        # Client projects
        "What clients are available for the capstone project?",
        "When do client projects officially start?",
        "Can students change their project team?",
    ],
    "output": [
        # Basic bootcamp information
        "This is a GenAI MLOps/LLMOps Bootcamp organized by WeCloudData and SDA (Saudi Digital Academy) in Riyadh.",
        "The bootcamp is located in Riyadh, Saudi Arabia.",
        "Week 0 sessions took place from Monday, August 25 to Thursday, August 28, 2025.",
        "Daily sessions start at 9:00 AM Riyadh time.",
        "Daily sessions end at 12:00 PM Riyadh time (with some afternoon sessions extending to 5:00 PM).",
        "The bootcamp operates in Riyadh Time timezone.",
        "Yes, attendance is mandatory and is taken daily.",
        "The main Zoom link is https://us02web.zoom.us/j/83424596656 with passcode 726039.",

        # Week 0 activities
        "Students were required to complete two surveys: a student background survey and an AI skills and goals survey.",
        "The student background survey helps understand students' background, motivations, and technical familiarity to tailor the bootcamp experience.",
        "The AI skills and goals survey assesses students' current AI skills and learning objectives.",
        "Students can submit questions or issues in a shared Google spreadsheet for office hour tickets.",
        "Week 0 sessions ran Monday through Thursday (Aug 25-28, 2025) from 9:00 AM to 12:00 PM.",

        # Project-related questions
        "The capstone project involves developing AI solutions in areas like Education, RFP, HR, or Enterprise AI, working in teams to solve real-world problems.",
        "Students started researching capstone projects on September 1, 2025 (Week 1).",
        "Project ideas include: AI for Education (instructors, students, schools), AI for Sales/RFP, AI for HR (workforce planning, hiring, job applications), Enterprise AI Hub, Apache Doris/VeloDB QA chatbot, and Text to SQL for real-time data warehouse.",
        "There were 10 project groups formed (Group 1 through Group 10).",
        "The draft project proposal and presentation were due at the end of Week 2, with presentations on Thursday of Week 2.",
        "Proposals should include: pain points, market research, user research via class survey, proposed solution (AI product description and implementation plan), and can be in Word doc or presentation format.",
        "Project proposals are submitted through a Google Form, with teams also maintaining collaborative Google Docs.",

        # Project ideas
        "AI for Education focuses on creating solutions for instructors, students, and schools to improve learning experiences.",
        "AI for RFP (Request for Proposal) aims to automate and streamline the RFP response process using AI.",
        "AI for HR includes workforce planning, hiring/recruiting, and job application assistance.",
        "Enterprise AI Hub focuses on secure deployment of LLM, Chat, Knowledge Search, and Workflows for enterprises.",
        "Apache Doris/VeloDB project involves creating a QA chatbot and Text to SQL functionality for real-time data warehouses.",

        # Technical topics covered
        "Week 1 covered Python fundamentals including functions, nested lists, list comprehension, and recursion.",
        "The Pandas module covers data manipulation, DataFrames, SQL-style joins, and data analysis techniques.",
        "Machine learning topics include regression, ensemble trees (bagging), hyperparameter tuning, and credit risk modeling.",
        "NLP topics include text preprocessing, tokenization, stemming, TF-IDF vectorization, sentiment analysis, and sentence embeddings.",
        "SentenceTransformer library (e.g., 'all-MiniLM-L6-v2' model) is used for creating sentence embeddings.",

        # Instructors and support
        "Instructors mentioned include WCD-Idris, shaohua-weclouddata, WCD-Carlo, Esteban (Gen AI - WCD), and Prabesh Lamichhane.",
        "Students can get help by using the office hour tickets spreadsheet, asking in class, or through breakout rooms during project sessions.",
        "Office hours can be accessed through the Week 0 Office Hour Tickets Google spreadsheet.",
        "Students provide instructor feedback through anonymous Google Forms surveys, conducted multiple times during the bootcamp.",

        # Schedule and holidays
        "Tuesday, September 23rd, 2025 was a holiday for Saudi National Day.",
        "Saudi National Day was celebrated on September 23rd, 2025.",
        "The capstone project officially starts in Week 9 or Week 10, though preparation began in Week 1.",

        # Technical tools and platforms
        "The learning portal at sda.weclouddata.com is used for accessing course materials, videos, and assignments.",
        "Tools used include Google Docs for collaboration, Miro boards for group ideation, Google Forms for surveys, and Zoom for virtual sessions.",
        "The Miro board is used for group assignment, project ideation, and organizing team information.",
        "Open data for Saudi Arabia can be found at https://data.gov.sa/ar, the National Data Bank portal.",

        # Code examples
        "You can flatten a nested list using recursion: check if each element is a list using isinstance(), recursively call the function on list elements, and append non-list elements to the result.",
        "To use OpenAI's API: import openai, create a client with openai.OpenAI(api_key=api_key), then call client.chat.completions.create() with model and messages parameters.",
        "Chat history format is a list of dictionaries with 'role' and 'content' keys: [{'role': 'user', 'content': 'question'}, {'role': 'assistant', 'content': 'answer'}].",
        "To create embeddings: from sentence_transformers import SentenceTransformer, load model with SentenceTransformer('all-MiniLM-L6-v2'), then call model.encode(sentences).",

        # Project validation
        "Teams should create user validation surveys, have classmates complete them, and track responses to validate their project ideas.",
        "User validation surveys should be created on Sunday (Week 2, Day 7) and distributed on Monday for completion.",
        "Teams submit and track user validation links in a shared Google spreadsheet.",

        # Presentation requirements
        "The project proposal presentation was held on Thursday of Week 2 (September 11, 2025).",
        "No, the first presentation in Week 2 was a practice presentation and was not graded.",
        "Students are expected to do 2-3 more presentations throughout the program, which will be graded.",

        # Model information
        "Claude Sonnet 4.5 is the smartest model in the Claude 4 family, efficient for everyday use, created by Anthropic.",
        "The Claude 4 family consists of Claude Opus 4.1, Claude Opus 4, Claude Sonnet 4.5, and Claude Sonnet 4.",
        "Claude can be accessed via web-based chat interface, mobile and desktop apps, API and developer platform (model string 'claude-sonnet-4-5-20250929'), and Claude Code command line tool.",

        # Additional resources
        "The book 'Build a Large Language Model (From Scratch)' by Sebastian Raschka is available as a shared PDF resource.",
        "Modal (modal.com) is a platform that provides free GPU access (smaller GPUs) for high-performance AI infrastructure and serverless computing.",
        "The Google Cloud ROI of AI 2025 report is available as a shared resource, emphasizing thinking about problems rather than just ideas.",

        # Client projects
        "Available clients include: WeCloudData (for Edtech AI projects), Beamdata (for MLOps/LLMOps projects), and VeloDB (starting Week 9).",
        "Client projects officially start in Week 9 or Week 10, though preparation and research begin earlier.",
        "Yes, students don't have to stay in the same project team for their final capstone project. Re-grouping happens when the real client project starts after Week 7.",
    ]
}

# ADD DATA AUGMENTATION: Create paraphrased versions
augmented_instructions = []
augmented_outputs = []

# Add paraphrased versions of key questions
paraphrases = {
    "What is the bootcamp about?": [
        "Tell me about this bootcamp",
        "Can you describe the bootcamp?",
        "What bootcamp is this?"
    ],
    "Where is the bootcamp located?": [
        "What is the location of the bootcamp?",
        "Where does the bootcamp take place?"
    ],
    "Is attendance mandatory?": [
        "Do I have to attend?",
        "Is it required to attend classes?"
    ],
    # Add more paraphrases for other important questions
}

# Add paraphrased versions
for orig_q, paraphrased_list in paraphrases.items():
    orig_idx = data["instruction"].index(orig_q)
    orig_answer = data["output"][orig_idx]
    for para_q in paraphrased_list:
        augmented_instructions.append(para_q)
        augmented_outputs.append(orig_answer)

# Combine original and augmented data
all_instructions = data["instruction"] + augmented_instructions
all_outputs = data["output"] + augmented_outputs

dataset = Dataset.from_dict({
    "instruction": all_instructions,
    "output": all_outputs
})

# Split into train/test
dataset = dataset.train_test_split(test_size=0.15, seed=42)

print(f"✅ Total samples: {len(all_instructions)}")
print(f"✅ Training samples: {len(dataset['train'])}")
print(f"✅ Test samples: {len(dataset['test'])}")

# ============================================================================
# 2. LOAD MODEL WITH 4-BIT QUANTIZATION (CRITICAL FOR MEMORY!)
# ============================================================================

from transformers import BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-1.5B-Instruct"  # Using larger model for better learning

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

# Set pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

# ============================================================================
# 3. COMPLETION-ONLY FORMATTING (KEY FOR KNOWLEDGE LEARNING!)
# ============================================================================

def format_instruction_completion_only(sample):
    """
    Format with special tokens so we only compute loss on the answer part.
    This is CRITICAL - we want the model to learn the answers, not repeat questions!
    """
    instruction = sample["instruction"]
    output = sample["output"]

    # Use chat template if available, otherwise custom format
    if hasattr(tokenizer, 'apply_chat_template'):
        messages = [
            {"role": "user", "content": instruction},
            {"role": "assistant", "content": output}
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    else:
        text = f"<|im_start|>user\n{instruction}<|im_end|>\n<|im_start|>assistant\n{output}<|im_end|>"

    return {"text": text}

# Apply formatting
dataset = dataset.map(format_instruction_completion_only, remove_columns=dataset["train"].column_names)

# Tokenize with response masking
def tokenize_with_labels(sample):
    """Tokenize and mask the instruction part so we only learn from the response"""
    full_text = sample["text"]

    # Tokenize the full text
    result = tokenizer(
        full_text,
        truncation=True,
        max_length=512,  # Increased for longer answers
        padding=False
    )

    # Find where the assistant response starts
    # We want to mask everything before the assistant's response
    input_ids = result["input_ids"]
    labels = input_ids.copy()

    # Find the assistant token to start computing loss from there
    # For Qwen, look for the assistant start token
    try:
        # Find where "<|im_start|>assistant" appears
        assistant_start_token = tokenizer.encode("<|im_start|>assistant", add_special_tokens=False)[0]
        if assistant_start_token in input_ids:
            assistant_idx = input_ids.index(assistant_start_token)
            # Mask everything before assistant response (set to -100)
            labels[:assistant_idx + 2] = [-100] * (assistant_idx + 2)  # +2 to skip the newline
    except:
        pass  # If we can't find it, compute loss on everything (less ideal but ok)

    result["labels"] = labels
    return result

tokenized_dataset = dataset.map(tokenize_with_labels, remove_columns=["text"], batched=False)

# ============================================================================
# 4. IMPROVED LoRA CONFIGURATION (MORE CAPACITY!)
# ============================================================================

lora_config = LoraConfig(
    r=32,  # INCREASED from 8 - more capacity to store knowledge!
    lora_alpha=64,  # INCREASED - controls how much LoRA affects the model
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention layers
        "gate_proj", "up_proj", "down_proj"  # MLP layers - CRITICAL for knowledge!
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ============================================================================
# 5. IMPROVED TRAINING CONFIGURATION
# ============================================================================

training_args = TrainingArguments(
    output_dir="./bootcamp_qwen_improved",
    num_train_epochs=20,  # MORE epochs for knowledge injection
    per_device_train_batch_size=2,  # Smaller batch for 4-bit
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,  # Effective batch size = 8
    learning_rate=5e-4,  # HIGHER learning rate for knowledge injection
    lr_scheduler_type="cosine",  # Better for longer training
    fp16=False,
    bf16=True,  # Better for 4-bit quantization
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    warmup_ratio=0.1,
    report_to="none",
    weight_decay=0.01,
    max_grad_norm=0.3,  # Prevent exploding gradients
    optim="paged_adamw_8bit",  # Memory efficient optimizer
)

# ============================================================================
# 6. CUSTOM DATA COLLATOR
# ============================================================================

from dataclasses import dataclass
from typing import Any, Dict, List

@dataclass
class DataCollatorForCompletionOnlyLM:
    tokenizer: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        input_ids = [f["input_ids"] for f in features]
        labels = [f["labels"] for f in features]

        max_length = max(len(ids) for ids in input_ids)

        padded_input_ids = []
        padded_labels = []
        attention_mask = []

        for ids, lbls in zip(input_ids, labels):
            padding_length = max_length - len(ids)
            padded_input_ids.append(ids + [self.tokenizer.pad_token_id] * padding_length)
            attention_mask.append([1] * len(ids) + [0] * padding_length)
            padded_labels.append(lbls + [-100] * padding_length)

        return {
            "input_ids": torch.tensor(padded_input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(padded_labels, dtype=torch.long)
        }

data_collator = DataCollatorForCompletionOnlyLM(tokenizer=tokenizer)

# ============================================================================
# 7. TRAIN WITH EVALUATION
# ============================================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
)

print("\n🚀 Starting IMPROVED training with knowledge injection...\n")
print("Key improvements:")
print("  ✅ 4-bit quantization for memory efficiency")
print("  ✅ Larger LoRA rank (r=32) for more capacity")
print("  ✅ Targeting MLP layers (critical for knowledge storage)")
print("  ✅ Completion-only training (only learn answers)")
print("  ✅ Higher learning rate (5e-4)")
print("  ✅ More epochs (20)")
print("  ✅ Data augmentation with paraphrases")
print()

trainer.train()

# ============================================================================
# 8. IMPROVED TESTING FUNCTION
# ============================================================================

def generate_response(question, max_new_tokens=150):
    """Generate response using chat template"""
    if hasattr(tokenizer, 'apply_chat_template'):
        messages = [{"role": "user", "content": question}]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        prompt = f"<|im_start|>user\n{question}<|im_end|>\n<|im_start|>assistant\n"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,  # Lower temperature for more factual answers
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract just the assistant's response
    if "<|im_start|>assistant" in response:
        response = response.split("<|im_start|>assistant")[-1].strip()
    if "<|im_end|>" in response:
        response = response.split("<|im_end|>")[0].strip()

    return response

# ============================================================================
# 9. COMPREHENSIVE TESTING
# ============================================================================

print("\n" + "="*80)
print("🎯 TESTING THE IMPROVED FINE-TUNED MODEL")
print("="*80 + "\n")

test_questions = [
    "What is the bootcamp about?",
    "Where is the bootcamp located?",
    "When was Saudi National Day?",
    "What Python topics were covered in Week 1?",
    "Who are the instructors?",
]

for question in test_questions:
    print(f"❓ Q: {question}")
    print(f"💡 A: {generate_response(question)}")
    print("-" * 80)

# ============================================================================
# 10. SAVE MODEL
# ============================================================================

model.save_pretrained("./bootcamp_qwen_final")
tokenizer.save_pretrained("./bootcamp_qwen_final")

print("\n✅ Model saved!")
print("\n📊 WHY THESE CHANGES HELP:")
print("1. LoRA rank 32 → More capacity to store new knowledge")
print("2. Target MLP layers → Where factual knowledge is stored")
print("3. Completion-only training → Model learns answers, not questions")
print("4. Higher LR + More epochs → Stronger knowledge injection")
print("5. 4-bit quantization → Train larger model in same memory")
print("6. Data augmentation → Model learns from paraphrases too")

✅ Total samples: 67
✅ Training samples: 56
✅ Test samples: 11


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 36,929,536 || all params: 1,580,643,840 || trainable%: 2.3364

🚀 Starting IMPROVED training with knowledge injection...

Key improvements:
  ✅ 4-bit quantization for memory efficiency
  ✅ Larger LoRA rank (r=32) for more capacity
  ✅ Targeting MLP layers (critical for knowledge storage)
  ✅ Completion-only training (only learn answers)
  ✅ Higher learning rate (5e-4)
  ✅ More epochs (20)
  ✅ Data augmentation with paraphrases



/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,3.394599,1.403888
2,1.438474,1.123387
3,0.674335,1.130680
4,0.384877,1.461457
5,0.195526,1.489258
6,0.144818,1.514910
7,0.152119,1.648758
8,0.116977,1.635873
9,0.109756,1.586260
10,0.091373,1.621078


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

In [9]:
test_questions = [
    "What is the bootcamp about?",
    "Where is the bootcamp located?",
    "When was Saudi National Day?",
    "What Python topics were covered in Week 1?",
    "Who are the instructors?",
]

for question in test_questions:
    print(f"❓ Q: {question}")
    print(f"💡 A: {generate_response(question)}")
    print("-" * 80)


❓ Q: What is the bootcamp about?
💡 A: system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
What is the bootcamp about?
assistant
This is a bootcamp organized by SDA and LLM for students in Riyadh to learn AI/LLMOps skills.
--------------------------------------------------------------------------------
❓ Q: Where is the bootcamp located?
💡 A: system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Where is the bootcamp located?
assistant
The bootcamp is located in Riyadh, Saudi Arabia.
--------------------------------------------------------------------------------
❓ Q: When was Saudi National Day?
💡 A: system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
When was Saudi National Day?
assistant
Saudi National Day is celebrated on June 23rd each year.
--------------------------------------------------------------------------------
❓ Q: What Python topics were covered in Week 1?
💡 A: system
You are Qwen, crea